In [4]:
import pandas as pd
import os 

In [38]:
def convert_id_to_str(df):
    """
    Convert the ID column of the in string
    """
    df = df.reset_index()
    df["ID"] = df["ID"].astype(str)
    df = df.set_index(df.columns[:2].tolist())
    return df

def rename_trials_to_average(df):
    for feature in [
        "PERCENT_SLOW_VITAL_CAPACITY_GLI_METHOD",
        "PERCENT_FORCED_VITAL_CAPACITY_GLI_METHOD",
    ]:
        # Find columns that match this feature pattern (like METHOD_1, METHOD_2 etc.)
        matching_cols = [col for col in df.columns if col.startswith(feature)]
        if len(matching_cols) == 1:
            method_1_col = matching_cols[0]
            avg_col_name = feature + "_AVERAGE"
            df = df.rename(columns={method_1_col: avg_col_name})
    return df

def compute_average_hhd(df):
    columns_hhd = [col for col in df.columns if col.startswith('HHD')]

    if columns_hhd:
        df['HHD_AVERAGE_TOTAL'] = df.groupby(['ID', 'TIME'])[columns_hhd].transform('mean').mean(axis=1)
    
    return df

def process_for_leaspy(df):
    df = convert_id_to_str(df)
    df = rename_trials_to_average(df)
    df = compute_average_hhd(df)
    # Only include patients who have at least 2 visits in ALSFRS-r
    #df_filtered = df.groupby("ID").filter(lambda x: x["ALSFRS_R_TOTAL"].notna().sum() > 1)
    return df

def process_for_survival_analysis(df, covariates):
    df = filter_on_covariates(df, covariates_to_filter=covariates)
    # Remove patients who died before baseline
    df = filter_dead_patients_before_baseline(df)
    return df 

def filter_on_covariates(df, covariates_to_filter):
    # Filter on available covariates (same as Trophos)
    df = df.dropna(subset=covariates_to_filter)
    # Filter on onset 
    df.loc[:, "ONSET_SYMPTOMS"] = df["ONSET_SYMPTOMS"].replace({
        "UPPER_SPINAL": "SPINAL",
        "LOWER_SPINAL": "SPINAL"
    })
    df = df[df["ONSET_SYMPTOMS"].isin(["BULBAR","SPINAL"])]    
    return df

def filter_dead_patients_before_baseline(df):
    df = df[
        (df["AGE_AT_DEATH"] >= df["AGE_AT_BASELINE"]) |
        (df["AGE_AT_DEATH"].isna())
    ]
    return df

# NEUROBANK

In [39]:
df_neurobank = pd.read_csv("_data/NEUROBANK_data.csv").set_index(["ID", "TIME"])
print(f"Processed: {len(df_neurobank.groupby('ID'))} patients, {len(df_neurobank)} visits")

Processed: 2620 patients, 17636 visits


### Data Corrections

In [40]:
covariates = [
    "AGE_AT_DIAGNOSIS",
    "SEX",
    "ONSET_SYMPTOMS",
    "ALSFRS_R_TOTAL",
    "BMI"
]
df_neurobank_leaspy_ready = process_for_leaspy(df_neurobank)
df_neurobank_survival_analysis_ready = process_for_survival_analysis(df_neurobank_leaspy_ready, covariates=covariates)
print(f"Processed: {len(df_neurobank_survival_analysis_ready.groupby('ID'))} patients, {len(df_neurobank_survival_analysis_ready)} visits")

Processed: 1706 patients, 6666 visits


# PROACT

In [41]:
df_proact = pd.read_csv("_data/PROACT_data.csv").set_index(["ID", "TIME"])
print(f"Processed: {len(df_proact.groupby('ID'))} patients, {len(df_proact)} visits")

Processed: 10106 patients, 104716 visits


### Data Corrections

In [42]:
#Fill ARM with Observational 
df_proact["ARM"] = df_proact["ARM"].fillna("Observational")

df_proact_leaspy_ready = process_for_leaspy(df_proact)
df_proact_survival_analysis_ready = process_for_survival_analysis(df_proact_leaspy_ready, covariates=covariates)
print(f"Processed: {len(df_proact_survival_analysis_ready.groupby('ID'))} patients, {len(df_proact_survival_analysis_ready)} visits")

Processed: 1644 patients, 6687 visits


# PULSE

In [43]:
df_pulse = pd.read_csv("_data/PULSE_data.csv").set_index(["ID", "TIME"])
df_pulse = df_pulse[df_pulse["ARM"]=="Case"]
print(f"Processed: {len(df_pulse.groupby('ID'))} patients, {len(df_pulse)} visits")

Processed: 497 patients, 2556 visits


### Data Corrections

In [45]:
df_pulse_leaspy_ready = process_for_leaspy(df_pulse)
df_pulse_survival_analysis_ready = process_for_survival_analysis(df_pulse_leaspy_ready, covariates=covariates)
print(f"Processed: {len(df_pulse_survival_analysis_ready.groupby('ID'))} patients, {len(df_pulse_survival_analysis_ready)} visits")

Processed: 457 patients, 2014 visits


# ANSWERALS

In [46]:
df_answerals = pd.read_csv("_data/ANSWERALS_data.csv").set_index(["ID", "TIME"])
df_answerals = df_answerals[df_answerals["ARM"]=="ALS"]
print(f"Processed: {len(df_answerals.groupby('ID'))} patients, {len(df_answerals)} visits")

Processed: 850 patients, 3152 visits


### Data Corrections

In [47]:
df_answerals_leaspy_ready = process_for_leaspy(df_answerals)
df_answerals_survival_analysis_ready = process_for_survival_analysis(df_answerals_leaspy_ready, covariates=covariates)
print(f"Processed: {len(df_answerals_survival_analysis_ready.groupby('ID'))} patients, {len(df_answerals_survival_analysis_ready)} visits")

Processed: 730 patients, 1979 visits


###

# TROPHOS

In [48]:
df_trophos = pd.read_csv("_data/TROPHOS_data.csv").set_index(["ID", "TIME"])
print(f"Processed: {len(df_trophos.groupby('ID'))} patients, {len(df_trophos)} visits")

Processed: 511 patients, 3066 visits


### Data Corrections

In [52]:
df_trophos_leaspy_ready = process_for_leaspy(df_trophos)
df_trophos_survival_analysis_ready = filter_on_covariates(df_trophos_leaspy_ready, covariates_to_filter=covariates)
print(f"Processed: {len(df_trophos_survival_analysis_ready.groupby('ID'))} patients, {len(df_trophos_survival_analysis_ready)} visits")

Processed: 508 patients, 2555 visits


## Save data


In [53]:
if not os.path.isdir("_data/leaspy_ready"):
    os.makedirs("_data/leaspy_ready")

df_trophos_leaspy_ready.to_csv("_data/leaspy_ready/TROPHOS_leaspy_ready.csv")
df_proact_leaspy_ready.to_csv("_data/leaspy_ready/PROACT_leaspy_ready.csv")
df_pulse_leaspy_ready.to_csv("_data/leaspy_ready/PULSE_leaspy_ready.csv")
df_answerals_leaspy_ready.to_csv("_data/leaspy_ready/ANSWERALS_leaspy_ready.csv")
df_neurobank_leaspy_ready.to_csv("_data/leaspy_ready/NEUROBANK_leaspy_ready.csv")

if not os.path.isdir("_data/survival_analysis_ready"):
    os.makedirs("_data/survival_analysis_ready")

df_trophos_survival_analysis_ready.to_csv("_data/survival_analysis_ready/TROPHOS_survival_analysis_ready.csv")
df_proact_survival_analysis_ready.to_csv("_data/survival_analysis_ready/PROACT_survival_analysis_ready.csv")
df_pulse_survival_analysis_ready.to_csv("_data/survival_analysis_ready/PULSE_survival_analysis_ready.csv")
df_answerals_survival_analysis_ready.to_csv("_data/survival_analysis_ready/ANSWERALS_survival_analysis_ready.csv")
df_neurobank_survival_analysis_ready.to_csv("_data/survival_analysis_ready/NEUROBANK_survival_analysis_ready.csv")

# All cohorts

In [15]:
df_trophos_leaspy_ready.loc[:,"COHORT"] = 'TROPHOS'
df_trophos_leaspy_ready.index = df_trophos_leaspy_ready.index.set_levels(
    df_trophos_leaspy_ready.index.levels[0] + "_TROPHOS", level=0
)

df_proact_leaspy_ready.loc[:,"COHORT"] = 'PROACT'
df_proact_leaspy_ready.index = df_proact_leaspy_ready.index.set_levels(
    df_proact_leaspy_ready.index.levels[0] + "_PROACT", level=0
)

df_pulse_leaspy_ready.loc[:,"COHORT"] = 'PULSE'
df_pulse_leaspy_ready.index = df_pulse_leaspy_ready.index.set_levels(
    df_pulse_leaspy_ready.index.levels[0] + "_PULSE", level=0
)

df_answerals_leaspy_ready.loc[:,"COHORT"] = 'ANSWERALS'
df_answerals_leaspy_ready.index = df_answerals_leaspy_ready.index.set_levels(
    df_answerals_leaspy_ready.index.levels[0] + "_ANSWERALS", level=0
)
df_neurobank_leaspy_ready.loc[:,"COHORT"] = 'NEUROBANK'
df_neurobank_leaspy_ready.index = df_neurobank_leaspy_ready.index.set_levels(
    df_neurobank_leaspy_ready.index.levels[0] + "_NEUROBANK", level=0
)


In [16]:
df_merged_leaspy_ready = (
    df_proact_leaspy_ready.reset_index()
    .merge(df_answerals_leaspy_ready.reset_index(), how="outer")
    .merge(df_trophos_leaspy_ready.reset_index(), how="outer")
    .merge(df_pulse_leaspy_ready.reset_index(), how="outer")
    .merge(df_neurobank_leaspy_ready.reset_index(), how="outer")
    .set_index(["ID","TIME"])
)

In [17]:
df_merged_ppct_leaspy_ready = (
    df_proact_leaspy_ready.reset_index()
    .merge(df_answerals_leaspy_ready.reset_index(), how="outer")
    .merge(df_pulse_leaspy_ready.reset_index(), how="outer")
    .merge(df_neurobank_leaspy_ready.reset_index(), how="outer")
    .set_index(["ID","TIME"])
)

In [18]:
df_merged_leaspy_ready.to_csv("_data/leaspy_ready/ALL_leaspy_ready.csv")

In [19]:
df_merged_ppct_leaspy_ready.to_csv("_data/leaspy_ready/ANSWERALS_PROACT_PULSE_NEUROBANK_leaspy_ready.csv")

###

## Filter to look like Trophos CT

### With Trophos inclusion criteria

- Patients aged 18–80 years 
- with El Escorial definite or probable ALS of between 6 and 36 months’ duration 
- who were treated with 50 mg riluzole twice a day for at least 1 month 
- slow vital capacity (SVC) of 70% or more

In [20]:
# Filter PROACT to look like CT
df_proact_leaspy_ready_ct = df_proact_leaspy_ready[df_proact_leaspy_ready['AGE_AT_BASELINE']<80]
print("Number of patients removed based on AGE:", df_proact_leaspy_ready.index.get_level_values("ID").nunique() - df_proact_leaspy_ready_ct.index.get_level_values("ID").nunique())

first_svc = df_proact_leaspy_ready_ct.groupby('ID')['PERCENT_SLOW_VITAL_CAPACITY_GLI_METHOD_AVERAGE'].first()
valid_ids = first_svc[first_svc >= 0.74].index # different normalization so threshold slightly different than inclusion criteria
df_proact_leaspy_ready_ct = df_proact_leaspy_ready_ct[df_proact_leaspy_ready_ct.index.get_level_values("ID").isin(valid_ids)]
print("Number of patients removed based on SVC:", df_proact_leaspy_ready.index.get_level_values("ID").nunique() - df_proact_leaspy_ready_ct.index.get_level_values("ID").nunique())

print("Final number of patients in PROACT dataset to mimic CT:", df_proact_leaspy_ready_ct.index.get_level_values("ID").nunique())
df_proact_leaspy_ready_ct.to_csv("_data/leaspy_ready/PROACT_CT_leaspy_ready.csv")

Number of patients removed based on AGE: 62
Number of patients removed based on SVC: 9949
Final number of patients in PROACT dataset to mimic CT: 157


In [21]:
# Filter ANSWERALS to look like CT
df_answerals_leaspy_ready_ct = df_answerals_leaspy_ready[df_answerals_leaspy_ready['AGE_AT_BASELINE']<80]
print("Number of patients removed based on AGE:", df_answerals_leaspy_ready.index.get_level_values("ID").nunique() - df_answerals_leaspy_ready_ct.index.get_level_values("ID").nunique())

first_svc = df_answerals_leaspy_ready_ct.groupby('ID')['PERCENT_SLOW_VITAL_CAPACITY_GLI_METHOD_AVERAGE'].first()
valid_ids = first_svc[first_svc >= 0.74].index # different normalization so threshold slightly different than inclusion criteria
df_answerals_leaspy_ready_ct = df_answerals_leaspy_ready_ct[df_answerals_leaspy_ready_ct.index.get_level_values("ID").isin(valid_ids)]
print("Number of patients removed based on SVC:", df_answerals_leaspy_ready.index.get_level_values("ID").nunique() - df_answerals_leaspy_ready_ct.index.get_level_values("ID").nunique())

print("Final number of patients in ANSWERALS dataset to mimic CT:", df_answerals_leaspy_ready_ct.index.get_level_values("ID").nunique())
df_answerals_leaspy_ready_ct.to_csv("_data/leaspy_ready/ANSWERALS_CT_leaspy_ready.csv")

Number of patients removed based on AGE: 22
Number of patients removed based on SVC: 418
Final number of patients in ANSWERALS dataset to mimic CT: 432


In [22]:
df_neurobank_leaspy_ready_ct = df_neurobank_leaspy_ready[df_neurobank_leaspy_ready['AGE_AT_BASELINE']<80]
print("Number of patients removed based on AGE:", df_neurobank_leaspy_ready.index.get_level_values("ID").nunique() - df_neurobank_leaspy_ready_ct.index.get_level_values("ID").nunique())

first_svc = df_neurobank_leaspy_ready_ct.groupby('ID')['PERCENT_SLOW_VITAL_CAPACITY_GLI_METHOD_AVERAGE'].first()
valid_ids = first_svc[first_svc >= 0.74].index # different normalization so threshold slightly different than inclusion criteria
df_neurobank_leaspy_ready_ct = df_neurobank_leaspy_ready_ct[df_neurobank_leaspy_ready_ct.index.get_level_values("ID").isin(valid_ids)]
print("Number of patients removed based on SVC:", df_neurobank_leaspy_ready.index.get_level_values("ID").nunique() - df_neurobank_leaspy_ready_ct.index.get_level_values("ID").nunique())

print("Final number of patients in NEUROBANK dataset to mimic CT:", df_neurobank_leaspy_ready_ct.index.get_level_values("ID").nunique())
df_neurobank_leaspy_ready_ct.to_csv("_data/leaspy_ready/NEUROBANK_CT_leaspy_ready.csv")


Number of patients removed based on AGE: 156
Number of patients removed based on SVC: 2373
Final number of patients in NEUROBANK dataset to mimic CT: 247


In [23]:
df_pulse_leaspy_ready_ct = df_pulse_leaspy_ready[df_pulse_leaspy_ready['AGE_AT_BASELINE']<80]
print("Number of patients removed based on AGE:", df_pulse_leaspy_ready.index.get_level_values("ID").nunique() - df_pulse_leaspy_ready_ct.index.get_level_values("ID").nunique())

first_svc = df_pulse_leaspy_ready_ct.groupby('ID')['PERCENT_SLOW_VITAL_CAPACITY_GLI_METHOD_AVERAGE'].first()
valid_ids = first_svc[first_svc >= 0.74].index # different normalization so threshold slightly different than inclusion criteria
df_pulse_leaspy_ready_ct = df_pulse_leaspy_ready_ct[df_pulse_leaspy_ready_ct.index.get_level_values("ID").isin(valid_ids)]
print("Number of patients removed based on SVC:", df_pulse_leaspy_ready.index.get_level_values("ID").nunique() - df_pulse_leaspy_ready_ct.index.get_level_values("ID").nunique())

print("Final number of patients in PULSE dataset to mimic CT:", df_pulse_leaspy_ready_ct.index.get_level_values("ID").nunique())
df_pulse_leaspy_ready_ct.to_csv("_data/leaspy_ready/PULSE_CT_leaspy_ready.csv")

Number of patients removed based on AGE: 27
Number of patients removed based on SVC: 139
Final number of patients in PULSE dataset to mimic CT: 358


In [24]:
df_mimic_ct_leaspy_ready = pd.concat([
    df_proact_leaspy_ready_ct,
    df_neurobank_leaspy_ready_ct,
    df_pulse_leaspy_ready_ct,
    df_answerals_leaspy_ready_ct,
])
df_mimic_ct_leaspy_ready.to_csv("_data/leaspy_ready/ANSWERALS_PROACT_PULSE_NEUROBANK_PULSE_CT_leaspy_ready.csv")


###